In [350]:
import numpy as np
from scipy import constants
import pandas as pd
import matplotlib.pyplot as plt
from tweezer_functions import * 
from IonChainTools import *
from scipy.optimize import fsolve
import matplotlib.colors as mcolors
import matplotlib.colorbar as mcolorbar
from scipy.optimize import fsolve
from scipy.optimize import curve_fit
from scipy.optimize import minimize
import matplotlib.ticker as ticker

#Constants in SI units
eps0 = constants.epsilon_0 
m = 39.9626*constants.atomic_mass
c = constants.c
e = constants.e
hbar = constants.hbar
pi = np.pi

# setting up parameters that we're not changing
qubit_wavelength = 729e-9
tweezer_wavelength = 532e-9
omega_tweezer = 2*pi*c/tweezer_wavelength
df = pd.read_csv("S_P_only.csv",sep = ",",encoding = "UTF-8")
lambdares = np.array(df["wavelength (nm)"])*1e-9
omega_res = 2*pi*c/lambdares
linewidths = np.array(df["A_ki (s^-1)"])
lifetimes = linewidths
print(linewidths)
#test

[1.47e+08 1.40e+08]


In [351]:
P_opt = 1 #W
w0 = 1e-6 #m
N = 3 #change me for a different number of ions

In [352]:
pot = potential(omega_tweezer,linewidths,omega_res,P_opt,w0)
w_tw_r = omega_tweezer_r(pot,w0,m) 
print("f_tw_r = ",w_tw_r/(2*pi*1e3),"kHz")

f_tw_r =  1009.8703958516896 kHz


In [353]:
f_rf_r = 1e6 #Hz
w_rf_r = f_rf_r*2*pi
ueq = ion_spacing(N,f_rf_r)[0]
f_rf_r_list = np.full(N,f_rf_r)
w_rf_r_list = np.full(N,w_rf_r)


In [354]:
eta(mode_calc_r(m, w_rf_r_list, ueq, N),729.147e-9,N)[0]

[0.140241262793494, 0.14024126279349566, 0.14024126279349583]

In [355]:
import itertools
import numpy as np
import pandas as pd

def tweezer_combos(w_tw_r, w_rf_r_list, m, ueq, mode_calc_r):
    rows = []   # we'll collect rows for the DataFrame
    n = len(w_rf_r_list)

    # Go through all possible subsets of positions (except all)
    for r in range(0, n):
        for positions in itertools.combinations(range(n), r):
            combo = []
            radial_freqs = []
            Mode0 = []
            Mode1 = []
            Mode2 = []
            for i in range(n):
                if i in positions:
                    combo.append(np.sqrt(w_tw_r**2 +  w_rf_r_list[i]**2))  # your formula
                else:
                    combo.append(w_rf_r_list[i])
            
            # compute radial modes
            tweezed_radial_modes = eta(mode_calc_r(m, combo, ueq, N),729.147e-9,N)
            Mode0.append(tweezed_radial_modes[0])
            Mode1.append(tweezed_radial_modes[1])
            Mode2.append(tweezed_radial_modes[2])

            for i in range(n):
                radial_freqs.append( mode_calc_r(m, combo, ueq, N)[i][0])


            # add one row to the DataFrame
            rows.append({
                "Ions tweezed": positions,
                "Combined radial frequencies": combo,
                "radial_modes": tweezed_radial_modes,
                "radial_freqs": radial_freqs,
                "Mode0": Mode0,
                "Mode1": Mode1,
                "Mode2": Mode2
            })

    # make into DataFrame
    return pd.DataFrame(rows)


In [356]:

df = tweezer_combos(w_tw_r, w_rf_r_list, m, ueq, mode_calc_r)

In [357]:
df


,Ions tweezed,Combined radial frequencies,radial_modes,radial_freqs,Mode0,Mode1,Mode2
0,(),"[6283185.307179586, 6283185.307179586, 6283185...","[[0.140241262793494, 0.14024126279349566, 0.14...","[999999.9999999999, 987253.616903689, 969127.0...","[[0.140241262793494, 0.14024126279349566, 0.14...","[[0.17286500236817037, -1.4762989882011933e-14...","[[-0.10073269475570484, 0.20146538951143378, -..."
1,"(0,)","[8929726.569216797, 6283185.307179586, 6283185...","[[-0.20428323217128508, -0.00399965085647122, ...","[1413309.936470321, 995129.8560200428, 972618....","[[-0.20428323217128508, -0.00399965085647122, ...","[[-0.003177714142035236, 0.13232604051601965, ...","[[0.0036625110616790107, -0.20670075305093552,..."
2,"(1,)","[6283185.307179586, 8929726.569216797, 6283185...","[[0.004146136718730079, 0.20468445097941076, 0...","[1407171.1684504698, 989401.237574516, 987253....","[[0.004146136718730079, 0.20468445097941076, 0...","[[0.17260647923353553, -0.006992715450698526, ...","[[-0.17286500236810806, -2.89128822563169e-15,..."
3,"(2,)","[6283185.307179586, 6283185.307179586, 8929726...","[[0.000586630624276605, 0.003999650856471827, ...","[1413309.9364703204, 995129.8560200421, 972618...","[[0.000586630624276605, 0.003999650856471827, ...","[[-0.20438031975550117, -0.13232604051600802, ...","[[0.13388534996343732, -0.20670075305094315, 0..."
4,"(0, 1)","[8929726.569216797, 8929726.569216797, 6283185...","[[-0.17020245981643833, -0.11241369065854599, ...","[1417908.283131464, 1402248.4878007069, 988326...","[[-0.17020245981643833, -0.11241369065854599, ...","[[0.11306958117398198, -0.1711205100641551, -0...","[[-0.0005086280969070579, -0.00492752201331596..."
5,"(0, 2)","[8929726.569216797, 6283185.307179586, 8929726...","[[0.14437091237747984, 0.005621270681399478, 0...","[1414341.8944172852, 1412270.484187453, 979123...","[[0.14437091237747984, 0.005621270681399478, 0...","[[-0.14453149730969564, 1.908983834914155e-15,...","[[-0.004777250364316141, 0.24538793196137393, ..."
6,"(1, 2)","[6283185.307179586, 8929726.569216797, 8929726...","[[0.002621898118464726, 0.11241369065855278, 0...","[1417908.2831314644, 1402248.4878007052, 98832...","[[0.002621898118464726, 0.11241369065855278, 0...","[[-0.0032162815489411205, -0.17112051006415144...","[[0.2442851169664839, -0.004927522013315223, -..."


In [358]:
np.sum(df["Mode0"][1])

-0.2088695136520329

In [359]:
df.insert(5, "COM vector sum", [np.abs(np.sum(df["Mode0"][i])) for i in range(len(df))])
df.insert(7, "Mode1 vector sum", [np.abs(np.sum(df["Mode1"][i])) for i in range(len(df))])
df.insert(9, "Mode2 vector sum", [np.abs(np.sum(df["Mode2"][i])) for i in range(len(df))])


In [363]:
df.sort_values(by=["Mode2 vector sum"], ascending=False, inplace=True)
df

,Ions tweezed,Combined radial frequencies,radial_modes,radial_freqs,Mode0,COM vector sum,Mode1,Mode1 vector sum,Mode2,Mode2 vector sum
6,"(1, 2)","[6283185.307179586, 8929726.569216797, 8929726...","[[0.002621898118464726, 0.11241369065855278, 0...","[1417908.2831314644, 1402248.4878007052, 98832...","[[0.002621898118464726, 0.11241369065855278, 0...",0.285238,"[[-0.0032162815489411205, -0.17112051006415144...",6.126721e-02,"[[0.2442851169664839, -0.004927522013315223, -...",2.388490e-01
4,"(0, 1)","[8929726.569216797, 8929726.569216797, 6283185...","[[-0.17020245981643833, -0.11241369065854599, ...","[1417908.283131464, 1402248.4878007069, 988326...","[[-0.17020245981643833, -0.11241369065854599, ...",0.285238,"[[0.11306958117398198, -0.1711205100641551, -0...",6.126721e-02,"[[-0.0005086280969070579, -0.00492752201331596...",2.388490e-01
5,"(0, 2)","[8929726.569216797, 6283185.307179586, 8929726...","[[0.14437091237747984, 0.005621270681399478, 0...","[1414341.8944172852, 1412270.484187453, 979123...","[[0.14437091237747984, 0.005621270681399478, 0...",0.294363,"[[-0.14453149730969564, 1.908983834914155e-15,...",7.432943e-14,"[[-0.004777250364316141, 0.24538793196137393, ...",2.358334e-01
3,"(2,)","[6283185.307179586, 6283185.307179586, 8929726...","[[0.000586630624276605, 0.003999650856471827, ...","[1413309.9364703204, 995129.8560200421, 972618...","[[0.000586630624276605, 0.003999650856471827, ...",0.208870,"[[-0.20438031975550117, -0.13232604051600802, ...",3.335286e-01,"[[0.13388534996343732, -0.20670075305094315, 0...",6.915289e-02
1,"(0,)","[8929726.569216797, 6283185.307179586, 6283185...","[[-0.20428323217128508, -0.00399965085647122, ...","[1413309.936470321, 995129.8560200428, 972618....","[[-0.20428323217128508, -0.00399965085647122, ...",0.208870,"[[-0.003177714142035236, 0.13232604051601965, ...",3.335286e-01,"[[0.0036625110616790107, -0.20670075305093552,...",6.915289e-02
2,"(1,)","[6283185.307179586, 8929726.569216797, 6283185...","[[0.004146136718730079, 0.20468445097941076, 0...","[1407171.1684504698, 989401.237574516, 987253....","[[0.004146136718730079, 0.20468445097941076, 0...",0.212977,"[[0.17260647923353553, -0.006992715450698526, ...",3.382202e-01,"[[-0.17286500236810806, -2.89128822563169e-15,...",1.081635e-13
0,(),"[6283185.307179586, 6283185.307179586, 6283185...","[[0.140241262793494, 0.14024126279349566, 0.14...","[999999.9999999999, 987253.616903689, 969127.0...","[[0.140241262793494, 0.14024126279349566, 0.14...",0.420724,"[[0.17286500236817037, -1.4762989882011933e-14...",1.110223e-15,"[[-0.10073269475570484, 0.20146538951143378, -...",7.216450e-16
